
# 12 — Gemini Video Understanding and Evidence Validation

**Learning reference:** [BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Adapted from:** `gemini_video_analysis.ipynb`

**Additional references:**
- [Gemini Video Understanding](https://ai.google.dev/gemini-api/docs/video-understanding)
- [Gemini Files API](https://ai.google.dev/gemini-api/docs/files)
- [Gemini Structured Outputs](https://ai.google.dev/gemini-api/docs/structured-output)

## Goal

Analyze a real video using Google's native Gemini SDK.

This notebook demonstrates:

1. Loading and previewing an MP4 video.
2. Uploading the video through the Gemini Files API.
3. Waiting until video processing is complete.
4. Extracting structured, timestamped observations.
5. Validating model responses using Pydantic.
6. Displaying a timeline of detected events.
7. Checking model-generated observations against the video.
8. Cleaning up uploaded resources.

Unlike earlier text-focused notebooks, this experiment
uses multimodal input: video containing visual and
potentially audio information.

The live analysis is performed in one Gemini request.
All subsequent validation and reporting run locally.

No real customer data or private recordings are used.



## 1. Understanding Multimodal AI

A modality is a type of information, such as text,
images, audio or video.

A multimodal model can process more than one type.

For example, a text-only model can answer questions
about a written description of a video.

A video-capable model can inspect the video itself.

In this experiment, we ask Gemini to identify
observable events and associate them with timestamps.

We distinguish three important concepts:

**Generation:** The model produces observations.

**Structural validation:** Python checks that the
response follows the expected data structure.

**Evidence validation:** A human checks whether the
reported observations actually appear in the video.

A correctly formatted response is not necessarily
a factually correct observation.


In [1]:

%pip install -q "google-genai==2.29.0" "pydantic>=2.9,<3" pandas


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [2]:

import json
import os
import shutil
import tempfile
import time

from getpass import getpass
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

from google import genai
from pydantic import BaseModel, Field, model_validator
from IPython.display import Video, display


MODEL = "gemini-3.8-flash"

SAMPLE_URL = (
    "https://storage.googleapis.com/"
    "generativeai-downloads/videos/Pottery.mp4"
)

VIDEO_PATH = ""

print("Model:", MODEL)
print("Setup complete.")
print("No Gemini requests made.")


Model: gemini-3.8-flash
Setup complete.
No Gemini requests made.



## 2. Load and Preview the Video

We use a public pottery video provided in Google's
example materials.

The video is downloaded into the temporary Colab
runtime, not into the GitHub repository.

We preview the video before asking Gemini to analyze it.

Watching the original video matters because it gives
us an independent source for verifying the model's
claims later.

This step does not call Gemini.

### Sample Video

This experiment uses Google's public pottery demonstration video.

**[▶ Watch the original pottery video](https://storage.googleapis.com/generativeai-downloads/videos/Pottery.mp4)**

The video is analyzed directly in Colab. GitHub displays
the notebook's saved analysis results, while the link above
provides access to the original footage for independent review.


In [5]:

if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser()

else:
    video_path = (
        Path(tempfile.gettempdir())
        / "gemini_video_pottery.mp4"
    )

    if not video_path.is_file():
        print("Downloading sample video...")

        temporary_path = video_path.with_suffix(".download")

        try:
            with urlopen(SAMPLE_URL, timeout=120) as source:
                with temporary_path.open("wb") as target:
                    shutil.copyfileobj(source, target)

            temporary_path.replace(video_path)

        finally:
            temporary_path.unlink(missing_ok=True)


if not video_path.is_file():
    raise FileNotFoundError(video_path)

if video_path.stat().st_size == 0:
    raise ValueError("Video file is empty.")

if video_path.suffix.lower() != ".mp4":
    raise ValueError("This example requires an MP4 video.")

print("Video:", video_path.name)
print(
    "Size:",
    round(video_path.stat().st_size / 1_000_000, 2),
    "MB"
)



from IPython.display import HTML

display(HTML(f'<video width="640" controls preload="metadata" src="{SAMPLE_URL}"></video>'))



Video: gemini_video_pottery.mp4
Size: 70.31 MB



## 3. Define the Structured Output

We want a short summary and a timeline of events.

Every event should contain:

- `start_seconds`: When the event begins.
- `end_seconds`: When the event ends.
- `visual_evidence`: What can actually be seen.
- `audio_evidence`: What can actually be heard, if relevant.
- `uncertainty`: Anything the video does not establish.

We use Pydantic to define these fields and validate
their types.

We also add a rule requiring every event's end time
to be later than its start time.

This is a structural check, not proof that the
event happened at the reported moment.


In [6]:

class VideoEvent(BaseModel):
    start_seconds: float = Field(
        ge=0,
        description="Start time in seconds."
    )

    end_seconds: float = Field(
        ge=0,
        description="End time in seconds."
    )

    visual_evidence: str = Field(
        min_length=1,
        description="What is directly visible."
    )

    audio_evidence: str | None = Field(
        description=(
            "What is directly audible, "
            "or null if not relevant."
        )
    )

    uncertainty: str | None = Field(
        description=(
            "Anything that cannot be confirmed, "
            "or null."
        )
    )

    @model_validator(mode="after")
    def validate_interval(self):
        if self.end_seconds <= self.start_seconds:
            raise ValueError(
                "Event end time must be after start time."
            )

        return self


class VideoAnalysis(BaseModel):
    title: str
    summary: str
    events: list[VideoEvent] = Field(
        min_length=1,
        max_length=3
    )


print("Pydantic schemas ready.")
print("No Gemini requests made.")


Pydantic schemas ready.
No Gemini requests made.


In [7]:

valid_event = VideoEvent(
    start_seconds=2,
    end_seconds=5,
    visual_evidence="A person moves an object.",
    audio_evidence=None,
    uncertainty=None
)

assert valid_event.start_seconds == 2

try:
    VideoEvent(
        start_seconds=8,
        end_seconds=3,
        visual_evidence="An object moves.",
        audio_evidence=None,
        uncertainty=None
    )

except ValueError:
    print("Invalid time interval correctly rejected.")

else:
    raise AssertionError(
        "The invalid interval should have failed."
    )

print("Offline validation tests passed.")


Invalid time interval correctly rejected.
Offline validation tests passed.



## 4. Upload the Video to Gemini

Gemini needs access to the video before it can analyze it.

The Files API uploads the MP4 and returns a file
reference containing information such as its URI.

Uploading and processing are separate stages.

A video may initially be in the `PROCESSING` state.

We must wait until the file reaches `ACTIVE` before
requesting analysis.

This upload operation does not generate a model answer.
The model analysis request comes later.


In [8]:

if "client" not in globals():
    gemini_key = getpass("Enter your Gemini API key: ")

    client = genai.Client(api_key=gemini_key)


if "video_file" not in globals():
    video_file = client.files.upload(
        file=str(video_path),
        config={
            "mime_type": "video/mp4",
            "display_name": video_path.name
        }
    )

    print("Video uploaded:", video_file.name)

else:
    print("Using existing uploaded video.")


video_file_name = video_file.name

if not video_file_name:
    raise RuntimeError("Upload returned no file name.")


deadline = time.monotonic() + 300

while (
    video_file.state is not None
    and video_file.state.name == "PROCESSING"
):
    if time.monotonic() >= deadline:
        raise TimeoutError(
            "Video processing did not finish. "
            "Do not upload a duplicate; check the "
            "existing file later."
        )

    time.sleep(3)

    video_file = client.files.get(
        name=video_file_name
    )


if (
    video_file.state is None
    or video_file.state.name != "ACTIVE"
):
    raise RuntimeError(
        f"Video is not ready: {video_file.state}"
    )


if not video_file.uri:
    raise RuntimeError("No video URI was returned.")


print("Video status:", video_file.state.name)
print("Ready for analysis.")
print("Model analysis requests: 0")


Enter your Gemini API key: ··········
Video uploaded: files/gzzk6x0otwqe
Video status: ACTIVE
Ready for analysis.
Model analysis requests: 0



## 5. Extract Timestamped Video Events

We now make one structured video-analysis request.

Gemini receives:

1. The uploaded video reference.
2. Instructions to focus on observable evidence.
3. A schema defining the expected JSON response.

The model should return:

- A short title.
- A concise video summary.
- One to three timestamped events.
- Visual evidence for each event.
- Audio evidence or uncertainty where relevant.

The response is validated locally before it is used
to build a timeline.

This experiment does not require a second model call
to judge the first model's response.


In [9]:

ANALYSIS_PROMPT = """
Analyze only the supplied video.

Return:
- A short title.
- A one-sentence summary.
- Between 1 and 3 important visible events.

For each event:
- Provide start and end times in seconds.
- Describe only what can actually be seen.
- Include audio evidence only if clearly audible.
- Record uncertainty rather than guessing.

Use timestamps relative to the beginning of the video.
Avoid invented events, identities or intentions.
"""


if "raw_video_response" not in globals():

    print("Sending one Gemini video-analysis request...")

    response = client.interactions.create(
        model=MODEL,
        input=[
            {
                "type": "video",
                "uri": video_file.uri,
                "mime_type": video_file.mime_type,
                "processing": "static",
                "resolution": "low"
            },
            {
                "type": "text",
                "text": ANALYSIS_PROMPT
            }
        ],
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": VideoAnalysis.model_json_schema()
        },
        generation_config={
            "max_output_tokens": 4096,
            "thinking_level": "low"
        },
        store=False,
        stream=False
    )

    if response.status != "completed":
        raise RuntimeError(
            f"Request status: {response.status}. "
            f"Errors: {response.errors}"
        )

    if not response.output_text:
        raise RuntimeError("Gemini returned no output text.")

    raw_video_response = response.output_text

    # Save the complete raw answer before validation.
    print("Gemini response received.")

else:
    print("Using previously saved Gemini response.")


print("\nRaw structured response:")
print(raw_video_response)


Sending one Gemini video-analysis request...
Gemini response received.

Raw structured response:
{
  "title": "Display of Handcrafted Pottery with Price Labels",
  "summary": "A camera pans over a table displaying ceramic tumblers and glazed bowls arranged next to handwritten descriptions and prices.",
  "events": [
    {
      "start_seconds": 0.0,
      "end_seconds": 6.0,
      "visual_evidence": "The camera moves closer to a stack and grouping of tumblers labeled '#5 Artichoke double dip TUMBLERS $20' and begins focusing on brownish glazed bowls labeled 'small bowls' with dimensions and price.",
      "audio_evidence": null,
      "uncertainty": null
    },
    {
      "start_seconds": 6.0,
      "end_seconds": 12.0,
      "visual_evidence": "The camera pans right across the bowls to show 'med bowls' labeled '4\"h x 7\"d $40' and a test tile labeled '#6 gemini double dip SLOW COOL'.",
      "audio_evidence": null,
      "uncertainty": null
    }
  ]
}



## 6. Validate the Model Response

Gemini has returned a structured response.

Now Python checks whether it matches our Pydantic
schema.

Validation checks include:

- Required fields are present.
- Timestamp values are non-negative numbers.
- Event descriptions are not empty.
- Every event ends after it starts.
- The number of events is between one and three.

These checks establish structural consistency.

They do not prove that the described events actually
occurred in the video.

That requires checking the source footage.


In [10]:

analysis = VideoAnalysis.model_validate_json(
    raw_video_response
)

print("Structural validation: PASSED")
print("Title:", analysis.title)
print("Events:", len(analysis.events))

print("\nValidated JSON:")
print(analysis.model_dump_json(indent=2))


Structural validation: PASSED
Title: Display of Handcrafted Pottery with Price Labels
Events: 2

Validated JSON:
{
  "title": "Display of Handcrafted Pottery with Price Labels",
  "summary": "A camera pans over a table displaying ceramic tumblers and glazed bowls arranged next to handwritten descriptions and prices.",
  "events": [
    {
      "start_seconds": 0.0,
      "end_seconds": 6.0,
      "visual_evidence": "The camera moves closer to a stack and grouping of tumblers labeled '#5 Artichoke double dip TUMBLERS $20' and begins focusing on brownish glazed bowls labeled 'small bowls' with dimensions and price.",
      "audio_evidence": null,
      "uncertainty": null
    },
    {
      "start_seconds": 6.0,
      "end_seconds": 12.0,
      "visual_evidence": "The camera pans right across the bowls to show 'med bowls' labeled '4\"h x 7\"d $40' and a test tile labeled '#6 gemini double dip SLOW COOL'.",
      "audio_evidence": null,
      "uncertainty": null
    }
  ]
}



## 7. Build a Video Timeline

Structured data is useful because we can work with
the events programmatically.

For example, we can:

- Sort events by their starting times.
- Display a timeline.
- Inspect visual and audio observations separately.
- Identify potentially invalid timestamps.

We will convert timestamps from numeric seconds
into the more familiar MM:SS format.

This table is generated by Python, not by an
additional language model request.


In [11]:

def format_timestamp(seconds):
    minutes, remaining = divmod(int(seconds), 60)
    return f"{minutes:02d}:{remaining:02d}"


sorted_events = sorted(
    analysis.events,
    key=lambda event: event.start_seconds
)


timeline_rows = []

for event in sorted_events:
    timeline_rows.append({
        "Start": format_timestamp(event.start_seconds),
        "End": format_timestamp(event.end_seconds),
        "Visual evidence": event.visual_evidence,
        "Audio evidence": event.audio_evidence,
        "Uncertainty": event.uncertainty
    })


timeline = pd.DataFrame(timeline_rows)

display(timeline)

print("Timeline generated locally.")
print("Additional Gemini requests: 0")


,Start,End,Visual evidence,Audio evidence,Uncertainty
0,00:00,00:06,The camera moves closer to a stack and groupin...,None,None
1,00:06,00:12,The camera pans right across the bowls to show...,None,None


Timeline generated locally.
Additional Gemini requests: 0



## 8. Evidence Verification

The model has generated a timeline, and Pydantic
has confirmed its structure.

We must now check whether its observations match
the source video.

For each event:

1. Locate the reported timestamp in the video.
2. Check whether the described action is visible.
3. Check whether any claimed audio evidence is audible.
4. Look for unsupported assumptions.
5. Record whether the evidence supports the observation.

A generated timestamp is a claim, not proof.

The verification below is a human assessment supported
by Python record keeping. It is not an automated
computer-vision accuracy test.


In [12]:

verification_records = []

for number, event in enumerate(sorted_events, start=1):
    verification_records.append({
        "event": number,
        "start": format_timestamp(event.start_seconds),
        "end": format_timestamp(event.end_seconds),
        "visual_claim": event.visual_evidence,
        "verified": None,
        "notes": "Not yet reviewed"
    })


verification_table = pd.DataFrame(
    verification_records
)

display(
    verification_table[
        ["event", "start", "end", "visual_claim",
         "verified", "notes"]
    ]
)

print(
    "Verification status: NOT YET COMPLETED"
)


,event,start,end,visual_claim,verified,notes
0,1,00:00,00:06,The camera moves closer to a stack and groupin...,None,Not yet reviewed
1,2,00:06,00:12,The camera pans right across the bowls to show...,None,Not yet reviewed


Verification status: NOT YET COMPLETED



### Manual Evidence Verification

The original pottery video was reviewed to verify
Gemini's observations and timestamps.

Both identified events were confirmed:

- **00:00–00:06:** The camera approaches ceramic
  tumblers and bowls.
- **00:06–00:12:** The camera shifts its focus
  toward medium bowls.

Although medium bowls are visible before 00:06,
the second event describes the camera's change
in focus rather than the objects' first appearance.

The observations are consistent with the video.


In [13]:

# Manual verification based on reviewing the original video.
# True = verified, False = incorrect, None = not checked.

manual_checks = {
    1: (
        True,
        "Ceramic tumblers and bowls are visible in the opening scene."
    ),
    2: (
        True,
        "Medium bowls are already visible at approximately 00:05, "
        "but from 00:06 the camera shifts to focus "
        "exclusively on them."
    )
}

for record in verification_records:
    event_number = record["event"]

    if event_number in manual_checks:
        verified, notes = manual_checks[event_number]

        if verified not in (True, False, None):
            raise ValueError("Invalid verification value.")

        record["verified"] = verified
        record["notes"] = notes

verification_table = pd.DataFrame(verification_records)

display(verification_table)

confirmed = sum(
    record["verified"] is True
    for record in verification_records
)

rejected = sum(
    record["verified"] is False
    for record in verification_records
)

unchecked = sum(
    record["verified"] is None
    for record in verification_records
)

print("Confirmed events:", confirmed)
print("Rejected events:", rejected)
print("Unchecked events:", unchecked)


,event,start,end,visual_claim,verified,notes
0,1,00:00,00:06,The camera moves closer to a stack and groupin...,True,Ceramic tumblers and bowls are visible in the ...
1,2,00:06,00:12,The camera pans right across the bowls to show...,True,Medium bowls are already visible at approximat...


Confirmed events: 2
Rejected events: 0
Unchecked events: 0



## 9. Clean Up Uploaded Resources

The video was uploaded through the Gemini Files API
for analysis.

After completing the analysis and evidence verification,
the uploaded file is no longer needed.

The following cell deletes the uploaded video from
the Gemini Files API and closes the API client.

This does not remove the original video from the local
Colab environment or erase the notebook's saved outputs.

The structured analysis and manual verification results
remain available in the notebook.


In [14]:

if "video_file_name" not in globals():
    print("No uploaded video reference found.")

elif globals().get("video_deleted", False):
    print("Uploaded video already deleted.")

else:
    client.files.delete(name=video_file_name)

    video_deleted = True

    print("Deleted uploaded video:", video_file_name)

client.close()

print("Gemini client closed.")
print("Cleanup complete.")


Deleted uploaded video: files/gzzk6x0otwqe
Gemini client closed.
Cleanup complete.



## Findings and Limitations

### What This Notebook Demonstrates

- Loading and previewing an MP4 video.
- Uploading video using Google's native Gemini SDK.
- Waiting until video processing is complete.
- Extracting structured, timestamped observations.
- Validating generated JSON using Pydantic.
- Creating a chronological timeline using Python.
- Manually verifying AI-generated observations.
- Cleaning up uploaded resources.

### Observed Results

- The sample pottery video was successfully uploaded
  and reached `ACTIVE` status.
- One Gemini video-analysis request completed successfully.
- Gemini generated a title, summary and two timestamped events.
- Both events passed Pydantic's structural validation.
- Python generated a chronological event timeline.
- Manual review confirmed both events and their
  corresponding camera movements.
- No discrepancies were identified in the two
  reviewed event descriptions.

### Manual Evidence Verification

The original pottery video was reviewed to verify
the model's observations and timestamps.

Both identified events were confirmed:

- **00:00–00:06:** The camera approaches ceramic
  tumblers and bowls.
- **00:06–00:12:** The camera shifts its focus toward
  medium bowls.

Although medium bowls are already visible before
00:06, the second event describes the camera's
change in focus rather than the objects'
first appearance.

Both observations are consistent with the video.

### Key Technical Concepts

**Multimodal understanding:** The model processes
video input rather than text alone.

**Files API:** Provides a reference to uploaded
media that can be passed to Gemini.

**Polling:** Checks whether the uploaded video
has finished processing before analysis begins.

**Structured output:** Requests information in a
predefined JSON format.

**Pydantic validation:** Checks data types,
required fields and timestamp ordering.

**Evidence verification:** Compares AI-generated
observations with the original video.

### Limitations

- Only one public demonstration video was analyzed.
- The results cannot establish general video-analysis
  accuracy across different videos.
- Video models may overlook brief visual events or
  estimate timestamps imprecisely.
- Valid JSON does not guarantee factual correctness.
- The timestamp validator checks event ordering but
  does not verify whether events fall within the
  video's actual duration.
- Manual verification involves human judgment.
- The experiment does not compare multiple models
  or repeated analysis runs.

### Conclusion

This notebook demonstrates a multimodal AI workflow
that converts video content into structured,
timestamped observations.

Gemini identified two events that passed both
Pydantic structural validation and manual
evidence verification.

The experiment illustrates how structured outputs,
Python validation and human review can work together
to assess AI-generated video observations.

Although both observations were confirmed in this
example, broader reliability would require testing
additional videos and more varied scenarios.
